<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/train_traditional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)
print("folder ready")

import sys
sys.path.append('/content/drive/MyDrive/ml_project')

Mounted at /content/drive
folder ready


In [2]:
%%writefile /content/drive/MyDrive/ml_project/train_traditional.py
# -*- coding: utf-8 -*-
import os
import pickle
import pandas as pd

from statsmodels.tsa.holtwinters import SimpleExpSmoothing
from statsmodels.tsa.arima.model import ARIMA

target_dir = "/content/drive/MyDrive/ml_project"

method_file = os.path.join(target_dir, "traditional_method.txt")
y_train_path = os.path.join(target_dir, "y_train.pkl")

model_output_path = os.path.join(target_dir, "traditional_trained.pkl")
info_output_path = os.path.join(target_dir, "traditional_info.txt")


# ---------------------------------------------------------
# train_traditional
# ---------------------------------------------------------
def train_traditional():

    if not os.path.exists(method_file):
        raise FileNotFoundError("traditional_method.txt not found")

    if not os.path.exists(y_train_path):
        raise FileNotFoundError("y_train.pkl not found")

    # load training series
    y_train = pd.read_pickle(y_train_path)
    y_series = pd.Series(y_train).dropna()

    # load selected model + parameters
    with open(method_file, "r") as f:
        lines = f.read().strip().split("\n")

    model_name = lines[0]
    params = lines[1] if len(lines) > 1 else ""

    training_info = []

    # ---------------------------------------------------------
    # naive (no training)
    # ---------------------------------------------------------
    if model_name == "naive":
        model = {"type": "naive"}
        training_info.append("model: naive")
        training_info.append("parameters: none")
        training_info.append("training: not required")

    # ---------------------------------------------------------
    # moving average (no training)
    # ---------------------------------------------------------
    elif model_name == "moving_average":
        window = int(params)
        model = {"type": "moving_average", "window": window}
        training_info.append("model: moving_average")
        training_info.append(f"parameters: window={window}")
        training_info.append("training: not required")

    # ---------------------------------------------------------
    # simple exponential smoothing
    # ---------------------------------------------------------
    elif model_name == "simple_exponential_smoothing":
        if params.strip() == "":
            model = SimpleExpSmoothing(y_series).fit()
            training_info.append("model: simple_exponential_smoothing")
            training_info.append("parameters: default (library)")
        else:
            alpha = float(params)
            model = SimpleExpSmoothing(y_series).fit(smoothing_level=alpha, optimized=False)
            training_info.append("model: simple_exponential_smoothing")
            training_info.append(f"parameters: alpha={alpha}")

        training_info.append("training: performed")

    # ---------------------------------------------------------
    # arima
    # ---------------------------------------------------------
    elif model_name == "arima":
        p, d, q = map(int, params.split(","))
        model = ARIMA(y_series, order=(p, d, q)).fit()

        training_info.append("model: arima")
        training_info.append(f"parameters: p={p}, d={d}, q={q}")
        training_info.append("training: performed")

    else:
        raise ValueError("unknown traditional model type")

    # save trained model
    with open(model_output_path, "wb") as f:
        pickle.dump(model, f)

    # save training info
    with open(info_output_path, "w") as f:
        for line in training_info:
            f.write(line + "\n")

    print("saved:", model_output_path)
    print("saved:", info_output_path)
    print("traditional model training completed")

    return model


Overwriting /content/drive/MyDrive/ml_project/train_traditional.py
